In [0]:
# ── Config ────────────────────────────────────────────────────────────
runs_table    = "test_automation_runs2"
results_table = "test_automation_results2"

from pyspark.sql.functions import col
import pandas as pd, json

NO_DATA_PATTERNS = [
    "NO RECORDS TO TEST", "NO MATCHING TEST DATA", "DOES NOT EXIST IN THE",
    "NO RECORDS FOUND", "NO DATA AVAILABLE", "NO DATA TO TEST",
    "NO DATA EXISTS FOR", "UNRESOLVED_COLUMN", "FAILED TO SETUP DATA",
]
ERROR_PATTERNS = ["IS NOT DEFINED", "TEST CRASHED:"]

def _reclassify(row):
    s = str(row["status"]).upper().strip()
    if s == "PASS":  return "PASS"
    if s in ("NO_DATA", "ERROR"): return s
    msg = str(row.get("message", "") or "").upper()
    for p in NO_DATA_PATTERNS:
        if p in msg: return "NO_DATA"
    for p in ERROR_PATTERNS:
        if p in msg: return "ERROR"
    return s

runs_df    = spark.table(runs_table)
results_df = spark.table(results_table)

trans_runs = (runs_df
              .filter(col("run_by_automation_name") == "Transformation_Tests")
              .orderBy("run_start_datetime")
              .toPandas())

if trans_runs.empty:
    displayHTML("<p>No runs found in table.</p>")
else:
    run_ids = trans_runs["run_id"].tolist()
    results = (results_df
               .filter(col("run_id").isin(run_ids))
               .select("run_id", "status", "message", "test_field", "test_from_state", "test_name")
               .toPandas())
    results["status_upper"] = results.apply(_reclassify, axis=1)
    run_meta = (trans_runs
                .set_index("run_id")[["run_start_datetime", "state_under_test"]]
                .to_dict("index"))

    # ── Per-run totals (for trend charts) ─────────────────────────────
    run_agg = []
    for rid, grp in results.groupby("run_id"):
        meta  = run_meta.get(rid, {})
        date  = str(meta.get("run_start_datetime", ""))[:10]
        state = str(meta.get("state_under_test", ""))
        p  = int((grp["status_upper"] == "PASS").sum())
        f  = int((grp["status_upper"] == "FAIL").sum())
        e  = int((grp["status_upper"] == "ERROR").sum())
        nd = int((grp["status_upper"] == "NO_DATA").sum())
        run_agg.append({
            "run_id": rid[:8], "date": date, "state": state,
            "pass": p, "fail": f, "error": e, "nodata": nd,
            "total": p + f + e + nd,
        })
    run_agg.sort(key=lambda x: x["date"])

    # ── Per-(from_state, field) per-run best status (for changes table) ─
    priority = {"PASS": 4, "FAIL": 3, "ERROR": 2, "NO_DATA": 1}
    field_runs = []
    for (from_state, field, rid), grp in results.groupby(["test_from_state", "test_field", "run_id"]):
        meta  = run_meta.get(rid, {})
        date  = str(meta.get("run_start_datetime", ""))[:10]
        state = str(meta.get("state_under_test", ""))
        if not date or not field or not from_state: continue
        best = max(grp["status_upper"].tolist(), key=lambda s: priority.get(s, 0))
        field_runs.append({
            "from_state": str(from_state), "field": str(field),
            "date": date, "run_state": state, "status": best,
        })
    field_runs.sort(key=lambda x: x["date"])

    all_states   = sorted(set(r["state"] for r in run_agg if r["state"]))
    data_json    = json.dumps(run_agg)
    states_json  = json.dumps(all_states)
    fr_json      = json.dumps(field_runs)

    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  body {{font-family:Arial,sans-serif;margin:0;padding:12px}}
  h2 {{color:#2c3e50;border-bottom:2px solid #2c3e50;padding-bottom:5px;margin-top:22px}}
  .boxes {{display:flex;gap:10px;flex-wrap:wrap;margin:10px 0}}
  .box {{padding:12px 18px;border-radius:6px;color:#fff;font-size:14px;text-align:center;min-width:80px}}
  .bp{{background:#27ae60}} .bf{{background:#e74c3c}} .be{{background:#7f8c8d}}
  .bn{{background:#f39c12}} .bt{{background:#2c3e50}}
  .fbar {{background:#ecf0f1;padding:10px 14px;border-radius:5px;margin:12px 0;
          display:flex;gap:12px;align-items:center;flex-wrap:wrap}}
  .fbar label {{font-weight:bold;color:#2c3e50;font-size:13px}}
  .bgrp {{display:flex;gap:4px}}
  .br {{padding:5px 12px;border:1px solid #bdc3c7;border-radius:3px;background:#fff;
        cursor:pointer;font-size:13px}}
  .br.active {{background:#2c3e50;color:#fff;border-color:#2c3e50}}
  select {{padding:6px 10px;border:1px solid #bdc3c7;border-radius:3px;font-size:13px;min-width:160px}}
  .crow {{display:flex;gap:20px;flex-wrap:wrap;margin:16px 0}}
  .cwrap {{flex:1;min-width:340px;background:#fafafa;border:1px solid #ddd;
           border-radius:5px;padding:12px}}
  .cwrap h3 {{margin:0 0 8px;font-size:14px;color:#2c3e50}}
  canvas {{max-height:280px}}
  table {{border-collapse:collapse;font-size:13px;width:100%;margin-top:8px}}
  th {{background:#2c3e50;color:#fff;padding:5px 10px;text-align:left}}
  td {{border:1px solid #ddd;padding:4px 10px}}
  tr:nth-child(even) {{background:#f9f9f9}}
  .gd{{color:#27ae60;font-weight:bold}} .rd{{color:#e74c3c;font-weight:bold}}
  .gr{{color:#7f8c8d}} .od{{color:#f39c12;font-weight:bold}}
  .ch-fixed {{background:#d5f5e3}} .ch-regr {{background:#fadbd8}}
  .ch-new {{background:#fef9e7}} .ch-still {{background:#fafafa}}
  .badge {{display:inline-block;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:bold;color:#fff}}
  .b-pass {{background:#27ae60}} .b-fail {{background:#e74c3c}}
  .b-err  {{background:#7f8c8d}} .b-nd   {{background:#f39c12}}
  .arrow  {{color:#888;font-size:16px;vertical-align:middle}}
  input[type=text] {{padding:6px 10px;border:1px solid #bdc3c7;border-radius:3px;font-size:13px;min-width:200px}}
</style></head><body>
<h2>Test Trends Report</h2>
<p id="sub" style="color:#666;font-size:13px"></p>

<div class="fbar">
  <label>Date range:</label>
  <div class="bgrp">
    <button class="br" onclick="setR(7)"  id="r7">Last 7 days</button>
    <button class="br" onclick="setR(30)" id="r30">Last 30 days</button>
    <button class="br active" onclick="setR(90)" id="r90">Last 90 days</button>
    <button class="br" onclick="setR(0)"  id="r0">All time</button>
  </div>
  <label>State:</label>
  <select id="sf" onchange="upd()"><option value="">All states</option></select>
</div>

<div class="boxes">
  <div class="box bp" id="bpass">PASS<br><b>-</b></div>
  <div class="box bf" id="bfail">FAIL<br><b>-</b></div>
  <div class="box be" id="berr">ERROR<br><b>-</b></div>
  <div class="box bn" id="bnd">NO DATA<br><b>-</b></div>
  <div class="box bt" id="btot">TOTAL<br><b>-</b></div>
</div>

<div class="crow">
  <div class="cwrap">
    <h3>Overall Results Over Time (daily totals)</h3>
    <canvas id="c1"></canvas>
  </div>
  <div class="cwrap">
    <h3>Failures by State Over Time</h3>
    <canvas id="c2"></canvas>
  </div>
</div>

<h2>Period Comparison</h2>
<p id="plabel" style="color:#666;font-size:13px"></p>
<table id="ptable" style="max-width:750px">
  <thead><tr>
    <th>State</th>
    <th>This period &#x2014; Fail</th><th>Prev period &#x2014; Fail</th>
    <th>Change &#x25BC;=fewer(good)</th>
    <th>This period &#x2014; Pass</th><th>Prev period &#x2014; Pass</th>
    <th>Change &#x25B2;=more(good)</th>
  </tr></thead>
  <tbody id="ptbody"></tbody>
</table>

<h2>Status Changes This Period vs Previous Period</h2>
<p style="color:#666;font-size:13px">
  Which (from-state, field) tests changed status between the two periods.
  <b class="gd">Fixed</b> = was FAIL/ERROR, now PASS.
  <b class="rd">Regressed</b> = was PASS, now FAIL/ERROR.
  <b class="od">New</b> = no data in previous period.
  <b class="gr">Still failing</b> = FAIL/ERROR in both.
</p>
<div class="fbar" style="margin-top:4px">
  <label>Filter field:</label>
  <input type="text" id="chsearch" oninput="renderChanges()" placeholder="Search field name..."/>
  <label>Change type:</label>
  <select id="chtype" onchange="renderChanges()">
    <option value="">All changes</option>
    <option value="fixed">Fixed</option>
    <option value="regressed">Regressed</option>
    <option value="new">New failure</option>
    <option value="still">Still failing</option>
  </select>
  <span id="ch-count" style="color:#7f8c8d;font-size:12px"></span>
</div>
<table>
  <thead><tr>
    <th>From State</th><th>Field</th>
    <th>Prev period</th><th></th><th>This period</th>
    <th>Change</th>
  </tr></thead>
  <tbody id="chtbody"></tbody>
</table>

<script>
var D  = {data_json};
var FR = {fr_json};
var STATES = {states_json};
var COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c',
              '#e67e22','#34495e','#e91e63','#00bcd4','#8bc34a','#ff5722',
              '#607d8b','#795548','#cddc39','#ff9800'];
var curR=90, ch1=null, ch2=null;
var _changeData = [];

function buildSel() {{
  var sel=document.getElementById('sf');
  STATES.forEach(function(s){{var o=document.createElement('option');o.value=s;o.textContent=s;sel.appendChild(o);}});
}}

function getCuts() {{
  if(curR<=0) return {{cur1:null,cur2:null,prev1:null,prev2:null}};
  var now=Date.now();
  return {{
    cur1:  new Date(now-curR*86400000).toISOString().slice(0,10),
    cur2:  null,
    prev1: new Date(now-curR*2*86400000).toISOString().slice(0,10),
    prev2: new Date(now-curR*86400000).toISOString().slice(0,10),
  }};
}}

function getFilt(cuts) {{
  var st=document.getElementById('sf').value;
  var d=D.filter(function(r){{return !st||r.state===st;}});
  if(cuts.cur1) d=d.filter(function(r){{return r.date>=cuts.cur1;}});
  return d;
}}

function getPrev(cuts) {{
  if(!cuts.prev1) return [];
  var st=document.getElementById('sf').value;
  return D.filter(function(r){{
    return (!st||r.state===st)&&r.date>=cuts.prev1&&r.date<cuts.prev2;
  }});
}}

function agg(data) {{
  var bd={{}};
  data.forEach(function(r){{
    if(!bd[r.date]) bd[r.date]={{pass:0,fail:0,error:0,nodata:0}};
    bd[r.date].pass+=r.pass; bd[r.date].fail+=r.fail;
    bd[r.date].error+=r.error; bd[r.date].nodata+=r.nodata;
  }});
  var dates=Object.keys(bd).sort();
  return {{dates:dates,bd:bd}};
}}

function aggState(data) {{
  var sts=[];
  data.forEach(function(r){{if(sts.indexOf(r.state)<0) sts.push(r.state);}});
  sts.sort();
  var bds={{}};
  data.forEach(function(r){{
    if(!bds[r.date]) bds[r.date]={{}};
    bds[r.date][r.state]=(bds[r.date][r.state]||0)+r.fail;
  }});
  var dates=Object.keys(bds).sort();
  return {{dates:dates,states:sts,bds:bds}};
}}

function delta(cur,prev,goodIsLess) {{
  if(prev===null||prev===undefined) return '<span class="gr">&#x2014;</span>';
  var d=cur-prev;
  if(d===0) return '<span class="gr">0</span>';
  var good=(d<0&&goodIsLess)||(d>0&&!goodIsLess);
  var arrow=d>0?'&#x25B2;':'&#x25BC;';
  return '<span class="'+(good?'gd':'rd')+'">'+arrow+' '+Math.abs(d)+'</span>';
}}

function badge(s) {{
  var cls={{PASS:'b-pass',FAIL:'b-fail',ERROR:'b-err',NO_DATA:'b-nd'}}[s]||'b-nd';
  return '<span class="badge '+cls+'">'+(s||'?')+'</span>';
}}

function sm(arr,k){{return arr.reduce(function(s,r){{return s+r[k];}},0);}}

var PRI = {{PASS:4,FAIL:3,ERROR:2,NO_DATA:1}};

function bestStatus(records) {{
  if(!records||!records.length) return null;
  return records.reduce(function(best,r){{
    return (PRI[r.status]||0)>(PRI[best.status]||0)?r:best;
  }}).status;
}}

function computeChanges(cuts) {{
  var st=document.getElementById('sf').value;
  var fd=FR.filter(function(r){{return !st||r.run_state===st;}});
  var curFD, prevFD;
  if(cuts.cur1) {{
    curFD  = fd.filter(function(r){{return r.date>=cuts.cur1;}});
    prevFD = fd.filter(function(r){{return cuts.prev1&&r.date>=cuts.prev1&&r.date<cuts.prev2;}});
  }} else {{
    curFD=fd; prevFD=[];
  }}

  var curBest={{}}, prevBest={{}};
  curFD.forEach(function(r){{
    var k=r.from_state+'||'+r.field;
    if(!curBest[k]||(PRI[r.status]||0)>(PRI[curBest[k]]||0)) curBest[k]=r.status;
  }});
  prevFD.forEach(function(r){{
    var k=r.from_state+'||'+r.field;
    if(!prevBest[k]||(PRI[r.status]||0)>(PRI[prevBest[k]]||0)) prevBest[k]=r.status;
  }});

  var all_keys=Object.keys(curBest);
  Object.keys(prevBest).forEach(function(k){{if(all_keys.indexOf(k)<0) all_keys.push(k);}});

  var changes=[];
  all_keys.forEach(function(k){{
    var parts=k.split('||'), from_state=parts[0], field=parts[1];
    var cur=curBest[k]||null, prev=prevBest[k]||null;
    var failing=function(s){{return s==='FAIL'||s==='ERROR';}};
    var passing=function(s){{return s==='PASS';}};
    var type;
    if(passing(cur)&&failing(prev))         type='fixed';
    else if(failing(cur)&&passing(prev))    type='regressed';
    else if(failing(cur)&&!prev)            type='new';
    else if(failing(cur)&&failing(prev))    type='still';
    else return;
    changes.push({{from_state:from_state,field:field,cur:cur,prev:prev,type:type}});
  }});

  var ord={{regressed:0,new:1,still:2,fixed:3}};
  changes.sort(function(a,b){{
    var d=ord[a.type]-ord[b.type]; if(d!==0) return d;
    return a.from_state<b.from_state?-1:a.from_state>b.from_state?1:0;
  }});
  return changes;
}}

function renderChanges() {{
  var search=(document.getElementById('chsearch').value||'').toLowerCase();
  var typeFilter=document.getElementById('chtype').value;
  var rows='', shown=0;
  var typeLabel={{fixed:'Fixed &#x2713;',regressed:'Regressed &#x26A0;',new:'New failure',still:'Still failing'}};
  var rowCls={{fixed:'ch-fixed',regressed:'ch-regr',new:'ch-new',still:'ch-still'}};

  _changeData.forEach(function(c){{
    if(typeFilter&&c.type!==typeFilter) return;
    if(search&&c.field.toLowerCase().indexOf(search)<0&&c.from_state.toLowerCase().indexOf(search)<0) return;
    shown++;
    rows+='<tr class="'+rowCls[c.type]+'">'
      +'<td>'+c.from_state+'</td>'
      +'<td>'+c.field+'</td>'
      +'<td>'+(c.prev?badge(c.prev):'<span class="gr">none</span>')+'</td>'
      +'<td class="arrow">&#x2192;</td>'
      +'<td>'+(c.cur?badge(c.cur):'<span class="gr">none</span>')+'</td>'
      +'<td><b>'+typeLabel[c.type]+'</b></td>'
      +'</tr>';
  }});
  document.getElementById('chtbody').innerHTML=rows||'<tr><td colspan="6" class="gr">No changes in this period.</td></tr>';
  document.getElementById('ch-count').textContent=shown+' of '+_changeData.length+' changes shown';
}}

function upd() {{
  var cuts=getCuts();
  var data=getFilt(cuts), prev=getPrev(cuts);
  var tp=sm(data,'pass'),tf=sm(data,'fail'),te=sm(data,'error'),tn=sm(data,'nodata'),tt=sm(data,'total');
  document.getElementById('bpass').innerHTML='PASS<br><b>'+tp+'</b>';
  document.getElementById('bfail').innerHTML='FAIL<br><b>'+tf+'</b>';
  document.getElementById('berr').innerHTML ='ERROR<br><b>'+te+'</b>';
  document.getElementById('bnd').innerHTML  ='NO DATA<br><b>'+tn+'</b>';
  document.getElementById('btot').innerHTML ='TOTAL<br><b>'+tt+'</b>';

  var a1=agg(data);
  var ds1=[
    {{label:'Pass',   data:a1.dates.map(function(d){{return a1.bd[d].pass;}}),   borderColor:'#27ae60',backgroundColor:'rgba(39,174,96,0.07)',  tension:0.3,fill:false,pointRadius:3}},
    {{label:'Fail',   data:a1.dates.map(function(d){{return a1.bd[d].fail;}}),   borderColor:'#e74c3c',backgroundColor:'rgba(231,76,60,0.07)',  tension:0.3,fill:false,pointRadius:3}},
    {{label:'Error',  data:a1.dates.map(function(d){{return a1.bd[d].error;}}),  borderColor:'#7f8c8d',backgroundColor:'rgba(127,140,141,0.07)',tension:0.3,fill:false,pointRadius:3}},
    {{label:'No Data',data:a1.dates.map(function(d){{return a1.bd[d].nodata;}}), borderColor:'#f39c12',backgroundColor:'rgba(243,156,18,0.07)', tension:0.3,fill:false,pointRadius:3}}
  ];
  if(ch1) ch1.destroy();
  ch1=new Chart(document.getElementById('c1'),{{
    type:'line',data:{{labels:a1.dates,datasets:ds1}},
    options:{{responsive:true,interaction:{{mode:'index',intersect:false}},
      plugins:{{legend:{{position:'bottom',labels:{{font:{{size:11}}}}}}}},
      scales:{{x:{{ticks:{{font:{{size:10}},maxRotation:45}}}},y:{{beginAtZero:true}}}}}}
  }});

  var a2=aggState(data);
  var ds2=a2.states.map(function(s,i){{
    return {{label:s,
      data:a2.dates.map(function(d){{return a2.bds[d]&&a2.bds[d][s]?a2.bds[d][s]:0;}}),
      borderColor:COLORS[i%COLORS.length],backgroundColor:'transparent',
      tension:0.3,fill:false,pointRadius:3}};
  }});
  if(ch2) ch2.destroy();
  ch2=new Chart(document.getElementById('c2'),{{
    type:'line',data:{{labels:a2.dates,datasets:ds2}},
    options:{{responsive:true,interaction:{{mode:'index',intersect:false}},
      plugins:{{legend:{{position:'bottom',labels:{{font:{{size:10}},boxWidth:10}}}}}},
      scales:{{x:{{ticks:{{font:{{size:10}},maxRotation:45}}}},y:{{beginAtZero:true,title:{{display:true,text:'Failures'}}}}}}}}
  }});

  var sf=document.getElementById('sf').value, sts=sf?[sf]:STATES;
  var rows='',ocf=0,opf=0,ocp=0,opp=0;
  sts.forEach(function(s){{
    var cr=data.filter(function(r){{return r.state===s;}}),
        pr=prev.filter(function(r){{return r.state===s;}});
    var cf=sm(cr,'fail'),pf=sm(pr,'fail'),cp=sm(cr,'pass'),pp=sm(pr,'pass');
    if(cf+pf+cp+pp===0) return;
    ocf+=cf; opf+=pf; ocp+=cp; opp+=pp;
    rows+='<tr><td><b>'+s+'</b></td>'+
      '<td class="rd">'+cf+'</td>'+
      '<td class="rd">'+(curR>0?pf:'&#x2014;')+'</td>'+
      '<td>'+(curR>0?delta(cf,pf,true):'&#x2014;')+'</td>'+
      '<td class="gd">'+cp+'</td>'+
      '<td class="gd">'+(curR>0?pp:'&#x2014;')+'</td>'+
      '<td>'+(curR>0?delta(cp,pp,false):'&#x2014;')+'</td></tr>';
  }});
  if(rows) rows='<tr style="background:#ecf0f1;font-weight:bold"><td>TOTAL</td>'+
    '<td class="rd">'+ocf+'</td><td class="rd">'+(curR>0?opf:'&#x2014;')+'</td>'+
    '<td>'+(curR>0?delta(ocf,opf,true):'&#x2014;')+'</td>'+
    '<td class="gd">'+ocp+'</td><td class="gd">'+(curR>0?opp:'&#x2014;')+'</td>'+
    '<td>'+(curR>0?delta(ocp,opp,false):'&#x2014;')+'</td></tr>'+rows;
  document.getElementById('ptbody').innerHTML=rows||'<tr><td colspan="7">No data in selected range.</td></tr>';

  var lbl=curR>0?'Last '+curR+' days':'All time';
  document.getElementById('sub').textContent=D.length+' runs total | '+lbl+' selected | '+data.length+' runs in view';
  document.getElementById('plabel').textContent=curR>0
    ?'Comparing last '+curR+' days vs previous '+curR+' days.  Green = improving, red = worsening.'
    :'All-time totals (no comparison available for all-time view).';

  _changeData=computeChanges(cuts);
  renderChanges();
}}

function setR(d) {{
  curR=d;
  ['r7','r30','r90','r0'].forEach(function(id){{document.getElementById(id).classList.remove('active');}});
  var m={{7:'r7',30:'r30',90:'r90',0:'r0'}};
  document.getElementById(m[d]).classList.add('active');
  upd();
}}

buildSel(); upd();
</script>
</body></html>"""
    displayHTML(html)
